In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: A Qiskit Function by Qedma

*Lihat [rujukan API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Fungsi Qiskit adalah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium IBM Quantum&reg;, Pelan Flex, dan Pelan On-Prem (melalui API Platform IBM Quantum). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.

## Gambaran keseluruhan
Walaupun unit pemprosesan kuantum telah bertambah baik dengan pesat dalam beberapa tahun kebelakangan ini, ralat akibat hingar dan ketidaksempurnaan perkakasan sedia ada tetap menjadi cabaran utama bagi pembangun algoritma kuantum. Apabila bidang ini mendekati pengiraan kuantum skala utiliti yang tidak boleh disahkan secara klasik, penyelesaian untuk membatalkan hingar dengan ketepatan yang terjamin semakin penting. Untuk mengatasi cabaran ini, Qedma telah membangunkan Quantum Error Mitigation (QESEM), yang disepadukan dengan lancar pada Platform IBM Quantum sebagai [Fungsi Qiskit](/guides/functions).

Dengan QESEM, pengguna boleh menjalankan litar kuantum mereka pada QPU berbising untuk mendapatkan keputusan tepat bebas ralat dengan overhed masa QPU yang sangat cekap, hampir kepada had asas. Untuk mencapai ini, QESEM memanfaatkan suite kaedah proprietari yang dibangunkan oleh Qedma untuk pencirian dan pengurangan ralat. Teknik pengurangan ralat merangkumi pengoptimuman gate, pentranspilan peka-hingar, penindasan ralat (ES), dan mitigasi ralat tidak berat sebelah (EM). Dengan gabungan kaedah berasaskan pencirian ini, pengguna boleh mencapai keputusan yang boleh dipercayai dan bebas ralat untuk litar kuantum bervolum besar yang generik, membuka kunci aplikasi yang tidak dapat dicapai sebaliknya.

Untuk penerangan penuh komponen asas, serta demonstrasi skala utiliti, rujuk kertas [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Penerangan
Anda boleh menggunakan fungsi QESEM oleh Qedma untuk menganggarkan dan melaksanakan litar anda dengan mudah menggunakan penindasan dan mitigasi ralat, mencapai volum litar yang lebih besar dan ketepatan yang lebih tinggi. Untuk menggunakan QESEM, anda menyediakan litar kuantum, satu set boleh cerap untuk diukur, ketepatan statistik sasaran untuk setiap boleh cerap, dan QPU yang dipilih. Sebelum menjalankan litar ke ketepatan sasaran, anda boleh menganggarkan masa QPU yang diperlukan berdasarkan pengiraan analitik yang tidak memerlukan pelaksanaan litar. Setelah anda berpuas hati dengan anggaran masa QPU, anda boleh melaksanakan litar dengan QESEM.

Apabila anda melaksanakan litar, QESEM menjalankan protokol pencirian peranti yang disesuaikan dengan litar anda, menghasilkan model hingar yang boleh dipercayai untuk ralat yang berlaku dalam litar. Berdasarkan pencirian, QESEM pertama melaksanakan pentranspilan peka-hingar untuk memetakan litar input ke set qubit dan gate fizikal, yang meminimumkan hingar yang mempengaruhi boleh cerap sasaran. Ini termasuk gate yang tersedia secara asli (CX/CZ pada peranti IBM&reg;), serta gate tambahan yang dioptimumkan oleh QESEM, membentuk set gate lanjutan QESEM. QESEM kemudian menjalankan set litar ES dan EM berasaskan pencirian pada QPU dan mengumpul hasil pengukuran mereka. Ini kemudian diproses secara klasik untuk memberikan nilai jangkaan tidak berat sebelah dan bar ralat untuk setiap boleh cerap, sepadan dengan ketepatan yang diminta.

![Gambaran keseluruhan Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM telah terbukti memberikan keputusan berketetapan tinggi untuk pelbagai aplikasi kuantum dan pada volum litar terbesar yang boleh dicapai hari ini. QESEM menawarkan ciri-ciri berikut untuk pengguna, yang ditunjukkan dalam bahagian penanda aras di bawah:
-	**Ketepatan terjamin:** QESEM mengeluarkan anggaran tidak berat sebelah untuk nilai jangkaan boleh cerap. Kaedah EM-nya dilengkapi dengan jaminan teori, yang - bersama dengan pencirian terkini Qedma - memastikan mitigasi menumpu ke output litar tanpa hingar sehingga ketepatan yang ditentukan pengguna. Berbanding dengan banyak kaedah EM heuristik yang terdedah kepada ralat sistematik atau berat sebelah, ketepatan terjamin QESEM penting untuk memastikan keputusan yang boleh dipercayai dalam litar dan boleh cerap kuantum yang generik.
-	**Penskalaan ke QPU besar:** Masa QPU QESEM bergantung pada volum litar, tetapi bebas daripada bilangan qubit. Qedma telah menunjukkan QESEM pada peranti kuantum terbesar yang tersedia hari ini, termasuk peranti IBM Quantum Eagle 127-qubit dan Heron 133-qubit.
-	**Bebas aplikasi:** QESEM telah ditunjukkan pada pelbagai aplikasi, termasuk simulasi Hamiltonian, VQE, QAOA, dan anggaran amplitud. Pengguna boleh memasukkan sebarang litar kuantum dan boleh cerap untuk diukur, dan mendapatkan keputusan tepat bebas ralat. Satu-satunya batasan didiktekan oleh spesifikasi perkakasan dan masa QPU yang diperuntukkan, yang menentukan volum litar yang boleh diakses dan ketepatan output. Berbanding, banyak penyelesaian pengurangan ralat adalah khusus untuk aplikasi atau melibatkan heuristik yang tidak terkawal, menjadikannya tidak terpakai untuk litar dan aplikasi kuantum generik.
-  **Set gate lanjutan:** QESEM menyokong gate dengan sudut pecahan, dan menyediakan gate $Rzz(\theta)$ sudut pecahan yang dioptimumkan Qedma pada peranti IBM Quantum Heron dan Eagle. Set gate lanjutan ini membolehkan kompilasi yang lebih cekap dan membuka kunci volum litar yang lebih besar sebanyak faktor 2 berbanding kompilasi CX/CZ lalai.
-	**Boleh cerap multiasas:** QESEM menyokong boleh cerap input yang terdiri daripada banyak rentetan Pauli yang tidak bertukar ganti, seperti Hamiltonian generik. Pilihan asas pengukuran dan pengoptimuman peruntukan sumber QPU (tembakan dan litar) kemudian dilakukan secara automatik oleh QESEM untuk meminimumkan masa QPU yang diperlukan untuk ketepatan yang diminta. Pengoptimuman ini, yang mengambil kira kesetiaan perkakasan dan kadar pelaksanaan, membolehkan anda menjalankan litar yang lebih dalam dan mendapatkan ketepatan yang lebih tinggi.
## Penanda aras
QESEM telah diuji pada pelbagai kes penggunaan dan aplikasi. Contoh-contoh berikut boleh membantu anda menilai jenis beban kerja yang boleh anda jalankan dengan QESEM.

Angka merit utama untuk mengukur kesukaran mitigasi ralat dan simulasi klasik untuk litar dan boleh cerap tertentu ialah **volum aktif**: bilangan gate CNOT yang mempengaruhi boleh cerap dalam litar. Volum aktif bergantung pada kedalaman dan lebar litar, pada wajaran boleh cerap, dan pada struktur litar, yang menentukan kon cahaya boleh cerap. Untuk maklumat lanjut, lihat ucapan dari [Sidang Kemuncak IBM Quantum 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM memberikan nilai yang sangat besar dalam rejim volum tinggi, memberikan keputusan yang boleh dipercayai untuk litar dan boleh cerap generik.

![Volum aktif](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplikasi    | Bilangan qubit | Peranti | Penerangan litar | Ketepatan | Jumlah masa | Penggunaan runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Litar VQE  | 8              | Eagle (r3) | 21 lapisan jumlah, 9 asas pengukuran, rantai 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 lapisan unik x 3 langkah, topologi hex-berat 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 lapisan unik x 8 langkah, topologi hex-berat 2D                     | 97%      | 116 min      | 23 min          |
| Simulasi Hamiltonian Trotterized   | 40  | Eagle (r3)            | 2 lapisan unik x 10 langkah Trotter, rantai 1D                    | 97%      | 3 jam     | 25 min         |
| Simulasi Hamiltonian Trotterized   | 119  | Eagle (r3)           | 3 lapisan unik x 9 langkah Trotter, topologi hex-berat 2D                    | 95%      | 6.5 jam     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 lapisan unik x 15 langkah, topologi hex-berat 2D                    | 99%      | 52 min             | 9 min           |

Ketepatan diukur di sini relatif terhadap nilai ideal boleh cerap: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, di mana '$\epsilon$' ialah ketepatan mutlak mitigasi (ditetapkan oleh input pengguna), dan $\langle O \rangle_{ideal}$ ialah boleh cerap pada litar tanpa hingar.
'Penggunaan runtime' mengukur penggunaan penanda aras dalam mod kelompok (jumlah penggunaan kerja individu), manakala 'jumlah masa' mengukur penggunaan dalam mod sesi (masa dinding eksperimen), yang merangkumi masa klasik dan komunikasi tambahan. QESEM tersedia untuk pelaksanaan dalam kedua-dua mod, supaya pengguna boleh memanfaatkan sebaik-baiknya sumber yang ada.

Litar Kicked Ising 28-qubit mensimulasikan Quasicrystal Masa Diskret yang dikaji oleh Shinjo et al. (lihat [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) dan [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) pada tiga gelung bersambung ibm_kawasaki. Parameter litar yang diambil di sini ialah $(\theta_x, \theta_z) = (0.9 \pi, 0)$, dengan keadaan awal feromagnet $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Boleh cerap yang diukur ialah nilai mutlak kemagnetan $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Eksperimen Kicked Ising skala utiliti dijalankan pada 136 qubit terbaik ibm_fez; penanda aras khusus ini dijalankan pada sudut Clifford $(\theta_x, \theta_z) = (\pi, 0)$, di mana volum aktif berkembang perlahan dengan kedalaman litar, yang - bersama dengan kesetiaan peranti yang tinggi - membolehkan ketepatan tinggi pada masa runtime yang singkat.

Litar simulasi Hamiltonian Trotterized adalah untuk model Ising Medan Transversal pada sudut pecahan: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ dan $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ secara berturutan (lihat [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Litar skala utiliti dijalankan pada 119 qubit terbaik ibm_brisbane, manakala eksperimen 40-qubit dijalankan pada rantai terbaik yang tersedia. Ketepatan dilaporkan untuk kemagnetan; keputusan berketetapan tinggi diperoleh untuk boleh cerap berwajaran lebih tinggi juga.

Litar VQE dibangunkan bersama penyelidik dari Pusat Teknologi dan Aplikasi Kuantum di Deutsches Elektronen-Synchrotron (DESY). Boleh cerap sasaran di sini ialah Hamiltonian yang terdiri daripada sejumlah besar rentetan Pauli yang tidak bertukar ganti, menekankan prestasi QESEM yang dioptimumkan untuk boleh cerap pelbagai asas. Mitigasi diterapkan pada ansatz yang dioptimumkan secara klasik; walaupun keputusan ini masih belum diterbitkan, keputusan dengan kualiti yang sama akan diperoleh untuk litar berbeza dengan sifat struktur yang serupa.
## Mulakan
Sahkan menggunakan [kunci API Platform IBM Quantum](http://quantum.cloud.ibm.com/) anda, dan pilih Fungsi Qiskit QESEM seperti berikut. (Petikan kod ini menganggap anda sudah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Contoh
Untuk bermula, cuba contoh asas ini untuk menganggarkan masa QPU yang diperlukan untuk menjalankan QESEM bagi `pub` tertentu:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Contoh berikut melaksanakan kerja QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Anda boleh menggunakan API Qiskit Serverless yang biasa untuk menyemak status beban kerja Fungsi Qiskit anda atau mendapatkan semula keputusan:

In [ ]:
print(sample_job.status())
result = sample_job.result()

Petikan kod berikut menunjukkan cara mendapatkan semula anggaran masa QPU (`estimate_time_only` ditetapkan):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

Petikan kod berikut menunjukkan cara mendapatkan semula keputusan mitigasi (apabila `estimate_time_only` tidak ditetapkan) dan metrik pelaksanaan. Ini mengandungi data penting yang membolehkan pemahaman lebih mendalam tentang bagaimana parameter berbeza mempengaruhi pelaksanaan QESEM. Ia juga mungkin relevan apabila menulis kertas berdasarkan penyelidikan anda.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Dapatkan mesej ralat
Jika status beban kerja anda adalah RALAT, gunakan `job.result()` untuk mendapatkan mesej ralat seperti berikut:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com.

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
- Visit the [API reference](/docs/api/functions/qedma-qesem) for this Qiskit Function.
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).


</Admonition>